In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_path = Path(".env")
load_dotenv(env_path)

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY :
    raise ValueError(f"YOUTUBE_API_KEY was not found. Check this file: {env_path.resolve()}")

In [ ]:
import pandas as pd
import requests

comment_url = "https://www.googleapis.com/youtube/v3/commentThreads"

from pathlib import Path
print(Path.cwd())

root_dir = Path.cwd().parent.parent

video_path = root_dir / "data" / "youtube_videos.csv"

video_csv = pd.read_csv(video_path)

# video_ids = video_csv['videoId']
# display(video_ids)

c:\Users\Playdata\Desktop\project1\crawling\youtube


,Unnamed: 0,videoId
0,0,oHLCAyjxywA
1,1,6nCTgra84_8
2,2,9mbc6ESMMvo
3,3,j__MUJTquJw
4,4,f3DLA1EU8vw
5,5,Y55mNGMowDw
6,6,xLPWWv_-gwI
7,7,jfmHPZofrFI
8,8,Kf3X8XakvHk
9,9,1-bqJRQ3XTs


In [ ]:
comment_items = []

# 채널별 동영상 저장된 csv에서 videoId 하나씩 순회
for video_id in video_csv['videoId']:
    # 영상마다 초기화 왜?
    page_token = None

    while True:
        comment_params = {
            "part": "snippet",
            "maxResults": 100,
            "videoId": video_id,
            "textFormat": "plainText",
            "key": YOUTUBE_API_KEY,
            "order": "relevance",
        }

        if page_token:
            comment_params["pageToken"] = page_token

        comment_response = requests.get(comment_url, params=comment_params)
        comment_response.raise_for_status()

        comment_data = comment_response.json()

        for item in comment_data.get("items", []):
            comment_list = item["snippet"]["topLevelComment"]["snippet"]
            # print(comment_list)

            comment_items.append(
                {
                    "videoId": comment_list["videoId"],
                    "text": comment_list["textDisplay"],
                    "likeCount": comment_list["likeCount"],
                    "published": comment_list["publishedAt"]
                }
            )

        page_token = comment_data.get("nextPageToken")

        if not page_token:
            break


In [27]:
comment_df = pd.DataFrame(comment_items)
display(comment_df.head(5))
print(comment_df.shape)

comment_csv = root_dir / "data" / "youtube_comment.csv"

comment_df.to_csv(comment_csv, encoding='utf-8')


,videoId,text,likeCount,published
0,oHLCAyjxywA,굳굳 1빠 잘보고있습니당,0,2026-08-11T05:06:59Z
1,oHLCAyjxywA,맑음님은 몇채널인지 공개하라 공개하라 !!!,2,2026-08-11T11:49:27Z
2,oHLCAyjxywA,다음 신규 보스 추가되면 수익 어떨지 궁금하네요!,0,2026-08-11T07:09:52Z
3,oHLCAyjxywA,달에 약 240만원정도 벌기 위해 총 큰 거 몇장정도 썼을려나 궁금하다,1,2026-08-11T06:47:48Z
4,oHLCAyjxywA,맑음님 정도 스팩 이면 상위 10위권 되나? 퍼클 에도 여러번 이름 올리니까,0,2026-08-11T07:45:21Z


(1960, 4)


In [34]:
keywords = [
    "캐릭터",
    "캐릭",
    "챌린저스",
    "챌섭",
    "챌썹",
    "200",
    "230",
    "260",
    "280",
    "레벨",
    "하이퍼버닝",
    "하이퍼 버닝",
    "하버",
]

question_words = [
    "왜",
    "어떻게",
    "뭐",
    "무엇",
    "어떤",
    "몇",
    "언제",
    "어디",
    "추천",
    "궁금",
]

question_endings = [
    "나요",
    "가요",
    "까요",
    "인가요",
    "되나요",
    "맞나요",
    "있나요",
    "없나요",
    "좋나요",
    "해야 하나요",
    "해야할까요",
]


def is_question(text):
    text = str(text).strip()

    has_question_mark = "?" in text or "？" in text

    has_question_word = any(word in text for word in question_words)

    has_question_ending = any(ending in text for ending in question_endings)

    return has_question_mark or has_question_word or has_question_ending


def has_keyword(text):
    text = str(text).lower()

    return any(keyword.lower() in text for keyword in keywords)


def is_target_comment(text):
    return is_question(text) and has_keyword(text)


mask = comment_df["text"].apply(is_target_comment)

filtered_df = comment_df[mask]

display(filtered_df.head(5))
print(filtered_df.shape)

filtered_path = root_dir / "data" / "filtered_youtube_comment.csv"

filtered_df.to_csv(filtered_path, encoding="utf-8")

,videoId,text,likeCount,published
75,9mbc6ESMMvo,지금 레벨 49 49 49 인데 오프라인 기준 경험치는 카 1-1 두면 되나요?,0,2026-08-06T10:45:55Z
76,9mbc6ESMMvo,맑음님 안녕하세요 지금 45레벨에 3-4까지 깬 상태인데 50레벨찍고 편하게 나머지...,0,2026-08-05T07:53:14Z
96,9mbc6ESMMvo,레벨 개높네 ㄷㄷ .. 43 44렙은 어디가야하지,0,2026-08-05T05:20:47Z
128,j__MUJTquJw,우선 지금까지 테스트한 내용입니다 (최종 확정)\r\n \r\n-----------...,25,2026-08-03T08:13:06Z
141,j__MUJTquJw,0:42 ㄹㅇㅋㅋ 진짜 울티마에서는 레벨만큼 중요한게 없음ㅋㅋㅋㅋ 암만 장비셋 잘해...,6,2026-08-02T11:22:20Z


(60, 4)
